# ⚽ Premier League Match Predictor
## Análisis de Datos y Predicción con Machine Learning

**Versión Final Mejorada** - Ejecuta en 8-10 minutos

### 📚 Contenido del Proyecto:
- ✅ Obtención de datos en tiempo real desde API
- ✅ Análisis exploratorio de datos (EDA)
- ✅ Ingeniería de características (Feature Engineering)
- ✅ Visualizaciones avanzadas
- ✅ Machine Learning (Random Forest)
- ✅ Evaluación de modelos
- ✅ **Interfaz interactiva con Gradio** 🌐
- ✅ Exportación de CSVs

---

### 🎯 Objetivo:
Crear un sistema inteligente que pueda **predecir el resultado de partidos** de la Premier League basándose en estadísticas actuales de los equipos, utilizando algoritmos de Machine Learning.

---

## 📦 SECCIÓN 1: INSTALACIÓN DE DEPENDENCIAS

### ¿Qué son las dependencias?
Son **librerías externas** (código escrito por otros desarrolladores) que nos permiten hacer tareas complejas sin escribir todo desde cero.

### Librerías que instalaremos:
- **pandas**: Manipulación y análisis de datos en tablas (DataFrames)
- **numpy**: Operaciones matemáticas y arrays numéricos
- **matplotlib**: Creación de gráficos y visualizaciones básicas
- **seaborn**: Visualizaciones estadísticas avanzadas (más bonitas)
- **scikit-learn**: Biblioteca de Machine Learning (modelos predictivos)
- **requests**: Para hacer peticiones HTTP a APIs
- **gradio**: Para crear interfaces web interactivas fácilmente

In [ ]:
# CELDA 1: INSTALACIÓN DE LIBRERÍAS
# El símbolo ! ejecuta comandos de terminal dentro de Jupyter/Colab
# pip install: comando para instalar paquetes de Python
# -q: significa "quiet" (silencioso), para mostrar menos texto

print("📦 Instalando dependencias...")
print("⏳ Esto puede tomar 30-60 segundos...\n")

!pip install -q pandas numpy matplotlib seaborn scikit-learn requests gradio

print("\n✅ Todas las dependencias instaladas correctamente")
print("🚀 Listo para comenzar el análisis")

## 📚 SECCIÓN 2: IMPORTACIÓN DE LIBRERÍAS

### ¿Qué es importar?
**Importar** significa cargar las librerías instaladas en la memoria para poder usarlas.

### Alias comunes:
- `import pandas as pd`: Usamos "pd" en lugar de escribir "pandas" cada vez
- `import numpy as np`: "np" es más corto que "numpy"
- `import matplotlib.pyplot as plt`: "plt" para crear gráficos

### Términos clave:
- **RandomForestClassifier**: Algoritmo de ML que usa múltiples "árboles de decisión"
- **train_test_split**: Función para dividir datos en entrenamiento y prueba
- **accuracy_score**: Mide qué tan preciso es nuestro modelo (% de aciertos)
- **confusion_matrix**: Tabla que muestra aciertos y errores del modelo
- **roc_curve, auc**: Métricas para evaluar la calidad del modelo

In [ ]:
# CELDA 2: IMPORTS Y CONFIGURACIÓN
# Importamos todas las librerías que vamos a necesitar

# Librerías para manejo de datos
import pandas as pd  # Tablas de datos (DataFrames)
import numpy as np   # Arrays y operaciones matemáticas

# Librerías para visualización
import matplotlib.pyplot as plt  # Gráficos básicos
import seaborn as sns           # Gráficos estadísticos elegantes

# Librerías de Machine Learning (scikit-learn)
from sklearn.ensemble import RandomForestClassifier  # Modelo Random Forest
from sklearn.model_selection import train_test_split  # Dividir datos
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc  # Métricas

# Otras utilidades
import requests  # Para hacer peticiones a APIs
import pickle    # Para guardar/cargar modelos entrenados
import os        # Para manejar archivos y carpetas
import warnings  # Para ocultar mensajes de advertencia
warnings.filterwarnings('ignore')  # Ignorar warnings molestos

# Configuración de estilos para gráficos
plt.style.use('seaborn-v0_8-darkgrid')  # Estilo visual de los gráficos
sns.set_palette("husl")                 # Paleta de colores

# Mostrar gráficos en el notebook
%matplotlib inline

# Crear carpetas para organizar archivos
# exist_ok=True: No dar error si la carpeta ya existe
os.makedirs('data', exist_ok=True)      # Para datos crudos
os.makedirs('outputs', exist_ok=True)   # Para resultados y CSVs
os.makedirs('models', exist_ok=True)    # Para modelos ML guardados

print("✅ Librerías importadas correctamente")
print("✅ Carpetas creadas: data/, outputs/, models/")
print("🎨 Estilo de gráficos configurado")
print("\n📂 Estructura de carpetas lista")

## 🌐 SECCIÓN 3: OBTENCIÓN DE DATOS DESDE API

### ¿Qué es una API?
**API** (Application Programming Interface) es un "puente" que permite a nuestro código comunicarse con servidores externos para obtener datos.

### Conceptos importantes:
- **Endpoint**: URL específica de la API (ej: `/standings` para tabla de posiciones)
- **API Key**: Clave secreta que nos identifica y autoriza a usar la API
- **Headers**: Información adicional que enviamos con la petición (como el API Key)
- **JSON**: Formato de datos que devuelve la API (similar a un diccionario de Python)
- **Timeout**: Tiempo máximo de espera antes de cancelar la petición

### Fuente de datos:
Usamos **football-data.org**, una API gratuita con datos actualizados de la Premier League.

In [ ]:
# CELDA 3: OBTENCIÓN DE DATOS DE LA API
# Conectamos con football-data.org para obtener datos actuales de la Premier League

# Configuración de la API
API_KEY = "f50c3bb69922405b8963e15c66c23877"  # Nuestra clave de acceso
LEAGUE = "PL"  # PL = Premier League
API_BASE = "https://api.football-data.org/v4"  # URL base de la API
HEADERS = {"X-Auth-Token": API_KEY}  # Headers con nuestra autenticación

print("📡 Conectando a la API de Premier League...")
print("🌍 Fuente: football-data.org")
print("⏳ Obteniendo tabla de posiciones actual...\n")

try:
    # Hacer petición GET a la API
    # GET: tipo de petición HTTP para OBTENER datos
    # timeout=10: esperar máximo 10 segundos
    response = requests.get(
        f"{API_BASE}/competitions/{LEAGUE}/standings",
        headers=HEADERS,
        timeout=10
    )
    
    # Convertir respuesta JSON a diccionario de Python
    data = response.json()
    
    # Extraer la tabla de posiciones
    # data["standings"][0]["table"]: navegamos por la estructura JSON
    table = data["standings"][0]["table"]
    
    # Procesar cada equipo y extraer información relevante
    teams = []
    for row in table:
        teams.append({
            "id": row["team"]["id"],                    # ID único del equipo
            "name": row["team"]["name"],                # Nombre del equipo
            "position": row["position"],                # Posición en la tabla
            "played": row["playedGames"],              # Partidos jugados
            "won": row["won"],                          # Partidos ganados
            "draw": row["draw"],                        # Empates
            "lost": row["lost"],                        # Derrotas
            "points": row["points"],                    # Puntos totales
            "goalsFor": row["goalsFor"],                # Goles a favor
            "goalsAgainst": row["goalsAgainst"],        # Goles en contra
            "goalDifference": row["goalDifference"]     # Diferencia de goles
        })
    
    # Crear DataFrame (tabla) con pandas
    # DataFrame: estructura de datos en forma de tabla (filas y columnas)
    df_teams = pd.DataFrame(teams)
    print(f"✅ Datos obtenidos exitosamente")
    print(f"📊 Total de equipos: {len(df_teams)}")
    
except Exception as e:
    # Si hay error (internet, API caída, etc), usar datos de ejemplo
    print(f"⚠️ Error conectando a la API: {e}")
    print("📂 Usando datos de ejemplo para demostración...\n")
    
    # Datos simulados basados en una temporada típica
    df_teams = pd.DataFrame({
        'id': list(range(1, 21)),
        'name': ['Arsenal FC', 'Liverpool FC', 'Manchester City FC', 'Tottenham Hotspur FC', 
                'Chelsea FC', 'Newcastle United FC', 'Manchester United FC', 'Brighton & Hove Albion FC',
                'Aston Villa FC', 'West Ham United FC', 'Crystal Palace FC', 'Brentford FC',
                'Fulham FC', 'Wolverhampton Wanderers FC', 'Everton FC', 'Nottingham Forest FC',
                'AFC Bournemouth', 'Luton Town FC', 'Burnley FC', 'Sheffield United FC'],
        'position': list(range(1, 21)),
        'played': [10] * 20,
        'won': [7, 6, 6, 5, 5, 4, 4, 3, 3, 3, 2, 2, 2, 1, 1, 1, 1, 0, 0, 0],
        'draw': [2, 3, 2, 3, 2, 4, 3, 5, 4, 3, 6, 5, 4, 7, 6, 5, 4, 8, 7, 6],
        'lost': [1, 1, 2, 2, 3, 2, 3, 2, 3, 4, 2, 3, 4, 2, 3, 4, 5, 2, 3, 4],
        'points': [23, 21, 20, 18, 17, 16, 15, 14, 13, 12, 12, 11, 10, 10, 9, 8, 7, 8, 7, 6],
        'goalsFor': [20, 18, 22, 16, 15, 14, 12, 13, 14, 11, 10, 12, 9, 8, 7, 9, 8, 6, 5, 4],
        'goalsAgainst': [8, 10, 9, 12, 13, 11, 14, 12, 15, 14, 12, 15, 13, 14, 16, 15, 18, 16, 19, 20],
        'goalDifference': [12, 8, 13, 4, 2, 3, -2, 1, -1, -3, -2, -3, -4, -6, -9, -6, -10, -10, -14, -16]
    })

# Guardar datos crudos en CSV
# index=False: no guardar el índice numérico de pandas
df_teams.to_csv('data/teams_raw.csv', index=False)
print("💾 Datos guardados en: data/teams_raw.csv")

# Mostrar primeros 5 equipos
print("\n📊 Tabla de Posiciones (Top 5):")
print("="*80)
df_teams[['position', 'name', 'played', 'points', 'goalsFor', 'goalsAgainst']].head()

## 📊 SECCIÓN 4: ANÁLISIS EXPLORATORIO DE DATOS (EDA)

### ¿Qué es EDA?
**EDA** (Exploratory Data Analysis) es el proceso de **explorar y entender** los datos antes de aplicar Machine Learning.

### ¿Por qué es importante?
- Identificar patrones y tendencias
- Detectar datos faltantes o errores
- Entender relaciones entre variables
- Obtener insights valiosos

### Métricas que calcularemos:
- **Media (mean)**: Promedio de los valores
- **Mediana (50%)**: Valor del medio
- **Desviación estándar (std)**: Cuánto varían los datos del promedio
- **Mínimo y Máximo**: Valores extremos

In [ ]:
# CELDA 4: ANÁLISIS EXPLORATORIO DE DATOS
# Exploramos los datos para entender su estructura y contenido

print("🔍 ANÁLISIS EXPLORATORIO DE DATOS")
print("="*80)

# 1. Información básica del dataset
print("\n📋 INFORMACIÓN GENERAL:")
print(f"   - Total de equipos: {len(df_teams)}")
print(f"   - Total de columnas: {len(df_teams.columns)}")
print(f"   - Columnas: {list(df_teams.columns)}")

# 2. Verificar datos faltantes
# .isnull(): retorna True si hay datos faltantes
# .sum(): cuenta cuántos True hay
missing_data = df_teams.isnull().sum()
print(f"\n   - Datos faltantes: {missing_data.sum()} (ninguno es ideal)")

# 3. Estadísticas descriptivas
# .describe(): calcula estadísticas básicas (media, std, min, max, etc)
print("\n📈 ESTADÍSTICAS DESCRIPTIVAS:")
print("="*80)
stats = df_teams[['points', 'goalsFor', 'goalsAgainst', 'won', 'draw', 'lost']].describe()
print(stats)

# 4. Top y bottom 3 equipos
print("\n🏆 TOP 3 EQUIPOS:")
print(df_teams[['position', 'name', 'points', 'goalsFor']].head(3))

print("\n⚠️ BOTTOM 3 EQUIPOS:")
print(df_teams[['position', 'name', 'points', 'goalsFor']].tail(3))

# 5. Insights interesantes
print("\n💡 INSIGHTS CLAVE:")
print(f"   - Equipo con más goles: {df_teams.loc[df_teams['goalsFor'].idxmax(), 'name']} ({df_teams['goalsFor'].max()} goles)")
print(f"   - Mejor defensa: {df_teams.loc[df_teams['goalsAgainst'].idxmin(), 'name']} ({df_teams['goalsAgainst'].min()} goles en contra)")
print(f"   - Promedio de puntos: {df_teams['points'].mean():.2f}")
print(f"   - Desviación estándar de puntos: {df_teams['points'].std():.2f}")

## ⚙️ SECCIÓN 5: FEATURE ENGINEERING (INGENIERÍA DE CARACTERÍSTICAS)

### ¿Qué es Feature Engineering?
Es el proceso de **crear nuevas variables** (features) a partir de los datos originales para mejorar el modelo de ML.

### Features que crearemos:

#### 1. **Métricas por Partido**:
- **points_per_game**: Puntos promedio por partido (mejor que puntos totales)
- **goals_for_per_game**: Goles anotados por partido
- **goals_against_per_game**: Goles recibidos por partido

#### 2. **Fuerzas del Equipo**:
- **attack_strength**: Potencia ofensiva (= goles por partido)
- **defense_strength**: Potencia defensiva (inverso de goles recibidos)

#### 3. **Eficiencia**:
- **win_rate**: Porcentaje de victorias
- **draw_rate**: Porcentaje de empates
- **loss_rate**: Porcentaje de derrotas

#### 4. **Scores Compuestos**:
- **form_score**: Combina puntos y diferencia de goles (forma actual)
- **team_score**: Score general (combina ataque + defensa + forma)

### ¿Por qué normalizar por partido?
No todos los equipos han jugado el mismo número de partidos, así que dividir por `played` hace las comparaciones más justas.

In [ ]:
# CELDA 5: FEATURE ENGINEERING
# Creamos nuevas características (features) para mejorar nuestro modelo ML

print("⚙️ CREANDO FEATURES AVANZADOS...")
print("="*80)

# Copiar DataFrame para no modificar el original
df_features = df_teams.copy()

# ==========================================
# 1. MÉTRICAS POR PARTIDO
# ==========================================
print("\n📊 Calculando métricas por partido...")

# .replace(0, 1): evita división por cero (si played=0, usa 1)
df_features['points_per_game'] = df_features['points'] / df_features['played'].replace(0, 1)
df_features['goals_for_per_game'] = df_features['goalsFor'] / df_features['played'].replace(0, 1)
df_features['goals_against_per_game'] = df_features['goalsAgainst'] / df_features['played'].replace(0, 1)

# ==========================================
# 2. FUERZA DE ATAQUE Y DEFENSA
# ==========================================
print("⚔️ Calculando fuerza de ataque y defensa...")

# Attack strength: simplemente goles por partido
df_features['attack_strength'] = df_features['goals_for_per_game']

# Defense strength: inverso de goles recibidos (menos goles = mejor defensa)
# Usamos 1/x para que valores bajos de goles en contra den valores ALTOS de defensa
# .replace(0, 0.1): evita división por cero
df_features['defense_strength'] = 1 / (df_features['goals_against_per_game'].replace(0, 0.1))

# ==========================================
# 3. TASAS DE RENDIMIENTO
# ==========================================
print("📈 Calculando tasas de victorias/empates/derrotas...")

# Porcentajes de resultados
df_features['win_rate'] = (df_features['won'] / df_features['played'].replace(0, 1)) * 100
df_features['draw_rate'] = (df_features['draw'] / df_features['played'].replace(0, 1)) * 100
df_features['loss_rate'] = (df_features['lost'] / df_features['played'].replace(0, 1)) * 100

# ==========================================
# 4. SCORES COMPUESTOS
# ==========================================
print("🎯 Calculando scores compuestos...")

# Diferencia de goles por partido
df_features['goal_difference_per_game'] = df_features['goalDifference'] / df_features['played'].replace(0, 1)

# Form Score: combina puntos y diferencia de goles
# 60% peso en puntos, 40% en diferencia de goles
df_features['form_score'] = (
    df_features['points_per_game'] * 0.6 + 
    df_features['goal_difference_per_game'] * 0.4
)

# Team Score: score general del equipo
# Combina ataque (35%) + defensa (35%) + forma (30%)
df_features['team_score'] = (
    df_features['attack_strength'] * 0.35 +
    df_features['defense_strength'] * 0.35 +
    df_features['form_score'] * 0.30
)

# Guardar DataFrame con features
df_features.to_csv('data/teams_with_features.csv', index=False)
print("\n✅ Features creados exitosamente")
print("💾 Guardado en: data/teams_with_features.csv")

# Mostrar top 5 equipos por score total
print("\n🏆 TOP 5 EQUIPOS POR SCORE TOTAL:")
print("="*80)
top_teams = df_features[['name', 'team_score', 'attack_strength', 'defense_strength', 'form_score']].sort_values('team_score', ascending=False).head()
print(top_teams)

# Explicar qué significa team_score
print("\n📌 NOTA: Team Score es un indicador compuesto que combina:")
print("   - Capacidad ofensiva (ataque)")
print("   - Capacidad defensiva (defensa)")
print("   - Rendimiento reciente (forma)")
print("   ➜ Valores más altos = equipos más fuertes")

## 📊 SECCIÓN 6: VISUALIZACIONES AVANZADAS

### ¿Por qué visualizar?
"Una imagen vale más que mil palabras". Las visualizaciones nos ayudan a:
- Identificar patrones rápidamente
- Comunicar insights de forma efectiva
- Detectar outliers (valores atípicos)
- Entender relaciones entre variables

### Tipos de gráficos que crearemos:
1. **Gráfico de barras horizontales**: Para comparar puntos entre equipos
2. **Histograma**: Para ver distribución de puntos
3. **Scatter plot**: Para ver relación ataque vs defensa
4. **Mapa de calor (heatmap)**: Para ver correlaciones entre variables

### Términos clave:
- **Correlación**: Relación entre dos variables (-1 a 1)
  - Positiva (+): cuando una sube, la otra también
  - Negativa (-): cuando una sube, la otra baja
  - Cero (0): no hay relación

In [ ]:
# CELDA 6: VISUALIZACIONES PRINCIPALES
# Creamos gráficos para entender mejor los datos

print("📊 GENERANDO VISUALIZACIONES...")
print("="*80)

# Crear figura con 4 subgráficos (2 filas x 2 columnas)
# figsize=(16, 12): tamaño en pulgadas (ancho, alto)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ==========================================
# GRÁFICO 1: PUNTOS POR EQUIPO
# ==========================================
print("📈 Creando gráfico de puntos...")

# Ordenar equipos por puntos (ascendente para que el primero quede arriba)
df_sorted = df_features.sort_values('points', ascending=True)

# Crear gráfico de barras horizontal
# axes[0, 0]: primer gráfico (fila 0, columna 0)
axes[0, 0].barh(df_sorted['name'], df_sorted['points'], color='steelblue')
axes[0, 0].set_xlabel('Puntos', fontsize=11)
axes[0, 0].set_title('Puntos por Equipo (2024-25)', fontsize=13, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)  # Grid vertical con transparencia

# ==========================================
# GRÁFICO 2: TOP 10 MEJORES ATAQUES
# ==========================================
print("⚔️ Creando gráfico de mejores ataques...")

# Seleccionar top 10 equipos con más goles
# .nlargest(10, 'goalsFor'): los 10 con más golesFor
top_attack = df_features.nlargest(10, 'goalsFor').sort_values('goalsFor')

axes[0, 1].barh(top_attack['name'], top_attack['goalsFor'], color='green', alpha=0.7)
axes[0, 1].set_xlabel('Goles a Favor', fontsize=11)
axes[0, 1].set_title('Top 10 Mejores Ataques', fontsize=13, fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# ==========================================
# GRÁFICO 3: SCATTER PLOT (ATAQUE VS DEFENSA)
# ==========================================
print("🎯 Creando scatter plot ataque vs defensa...")

# Scatter: cada punto es un equipo
# x = goles a favor, y = goles en contra
# s = tamaño de puntos basado en puntos totales
# c = color basado en team_score
scatter = axes[1, 0].scatter(
    df_features['goalsFor'],
    df_features['goalsAgainst'],
    s=df_features['points'] * 10,  # Tamaño proporcional a puntos
    c=df_features['team_score'],   # Color según team_score
    cmap='viridis',                # Paleta de colores
    alpha=0.6,                     # Transparencia
    edgecolors='black'
)
axes[1, 0].set_xlabel('Goles a Favor (Ataque)', fontsize=11)
axes[1, 0].set_ylabel('Goles en Contra (Defensa)', fontsize=11)
axes[1, 0].set_title('Ataque vs Defensa', fontsize=13, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Añadir colorbar (leyenda de colores)
plt.colorbar(scatter, ax=axes[1, 0], label='Team Score')

# ==========================================
# GRÁFICO 4: MATRIZ DE CORRELACIÓN
# ==========================================
print("🔥 Creando matriz de correlación...")

# Seleccionar columnas numéricas para correlación
numeric_cols = ['points', 'goalsFor', 'goalsAgainst', 'won', 'draw', 'lost']

# .corr(): calcula correlación entre todas las columnas
# Valores: -1 (correlación negativa) a +1 (correlación positiva)
corr = df_features[numeric_cols].corr()

# Crear heatmap (mapa de calor)
# annot=True: mostrar valores numéricos
# fmt='.2f': formato con 2 decimales
# cmap='coolwarm': colores frío-calor
# center=0: centrar escala en 0
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    ax=axes[1, 1],
    center=0,
    square=True,
    linewidths=1
)
axes[1, 1].set_title('Matriz de Correlación', fontsize=13, fontweight='bold')

# Ajustar espaciado entre gráficos
plt.tight_layout()

# Guardar imagen en alta resolución
# dpi=150: puntos por pulgada (mayor = mejor calidad)
# bbox_inches='tight': ajustar bordes
plt.savefig('outputs/visualizaciones_principales.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualizaciones creadas y guardadas")
print("💾 Archivo: outputs/visualizaciones_principales.png")

# Explicar insights del scatter plot
print("\n💡 INSIGHT DEL SCATTER PLOT:")
print("   - Esquina superior izquierda: Buen ataque, mala defensa")
print("   - Esquina inferior derecha: Mal ataque, buena defensa")
print("   - Esquina inferior izquierda: Mal ataque, mala defensa")
print("   - Los MEJORES equipos tienen: muchos goles a favor + pocos en contra")

## 🤖 SECCIÓN 7: PREPARACIÓN DE DATOS PARA MACHINE LEARNING

### ¿Cómo funciona nuestro predictor?
Para predecir quién gana entre Equipo A vs Equipo B, creamos **características diferenciales**:

### Features que usaremos:
1. **attack_diff**: Diferencia de ataque (A - B)
2. **defense_diff**: Diferencia de defensa (A - B)
3. **form_diff**: Diferencia de forma (A - B)
4. **points_diff**: Diferencia de puntos (A - B)
5. **goal_diff_diff**: Diferencia de diferencia de goles (A - B)
6. **teamA_attack**: Ataque absoluto de A
7. **teamA_defense**: Defensa absoluta de A
8. **teamA_form**: Forma absoluta de A
9. **teamB_attack**: Ataque absoluto de B
10. **teamB_defense**: Defensa absoluta de B
11. **teamB_form**: Forma absoluta de B
12. **score_diff**: Diferencia de score total (A - B)

### Variable objetivo (label):
- **winner**: 1 si gana Team A, 0 si gana Team B

### ¿Cómo determinamos el ganador?
Usamos una **función sigmoide** basada en score_diff:
- Si score_diff > 0 (A es mejor) → mayor probabilidad de que A gane
- Si score_diff < 0 (B es mejor) → mayor probabilidad de que B gane

In [ ]:
# CELDA 7: CREACIÓN DEL DATASET DE ENTRENAMIENTO
# Creamos todas las combinaciones posibles de partidos entre equipos

print("🤖 CREANDO DATASET PARA MACHINE LEARNING...")
print("="*80)

matches = []  # Lista para almacenar todos los partidos

# Doble bucle: probar cada equipo contra cada otro equipo
# i, j son índices; team_a, team_b son las filas completas
for i, team_a in df_features.iterrows():
    for j, team_b in df_features.iterrows():
        # Solo crear partido si son equipos diferentes
        if i != j:
            # Crear diccionario con todas las características del partido
            match = {
                # IDs y nombres
                'teamA_id': team_a['id'],
                'teamB_id': team_b['id'],
                'teamA_name': team_a['name'],
                'teamB_name': team_b['name'],
                
                # Diferencias relativas (A - B)
                'attack_diff': team_a['attack_strength'] - team_b['attack_strength'],
                'defense_diff': team_a['defense_strength'] - team_b['defense_strength'],
                'form_diff': team_a['form_score'] - team_b['form_score'],
                'points_diff': team_a['points'] - team_b['points'],
                'goal_diff_diff': team_a['goalDifference'] - team_b['goalDifference'],
                
                # Características absolutas de Team A
                'teamA_attack': team_a['attack_strength'],
                'teamA_defense': team_a['defense_strength'],
                'teamA_form': team_a['form_score'],
                
                # Características absolutas de Team B
                'teamB_attack': team_b['attack_strength'],
                'teamB_defense': team_b['defense_strength'],
                'teamB_form': team_b['form_score'],
                
                # Scores totales
                'teamA_score': team_a['team_score'],
                'teamB_score': team_b['team_score'],
                'score_diff': team_a['team_score'] - team_b['team_score'],
            }
            
            # Calcular probabilidad de que Team A gane usando función sigmoide
            # Fórmula: P(A gana) = 1 / (1 + e^(-score_diff))
            # np.exp(): función exponencial (e^x)
            prob_a_wins = 1 / (1 + np.exp(-match['score_diff']))
            
            # Asignar winner: 1 si A gana (prob > 50%), 0 si B gana
            match['winner'] = 1 if prob_a_wins > 0.5 else 0
            
            # Agregar partido a la lista
            matches.append(match)

# Convertir lista de diccionarios a DataFrame
df_matches = pd.DataFrame(matches)

# Guardar dataset
df_matches.to_csv('data/matches_dataset.csv', index=False)

print(f"✅ Dataset de partidos creado exitosamente")
print(f"\n📊 ESTADÍSTICAS DEL DATASET:")
print(f"   - Total de enfrentamientos posibles: {len(df_matches)}")
print(f"   - Victorias de Team A: {(df_matches['winner'] == 1).sum()}")
print(f"   - Victorias de Team B: {(df_matches['winner'] == 0).sum()}")
print(f"   - Features por partido: {len(match) - 1}")

# Mostrar ejemplo de partido
print(f"\n🧪 EJEMPLO DE PARTIDO:")
example_match = df_matches.iloc[0]
print(f"   {example_match['teamA_name']} vs {example_match['teamB_name']}")
print(f"   Ganador predicho: {'Team A' if example_match['winner'] == 1 else 'Team B'}")
print(f"   Score diff: {example_match['score_diff']:.2f}")

print(f"\n💾 Dataset guardado en: data/matches_dataset.csv")

## 🧠 SECCIÓN 8: ENTRENAMIENTO DEL MODELO DE MACHINE LEARNING

### ¿Qué es Random Forest?
**Random Forest** (Bosque Aleatorio) es un algoritmo de ML que funciona así:
1. Crea muchos **árboles de decisión** (de ahí "bosque")
2. Cada árbol hace una predicción
3. La predicción final es el **voto mayoritario** de todos los árboles

### ¿Por qué Random Forest?
- ✅ Muy preciso y robusto
- ✅ Maneja bien datos no lineales
- ✅ Menos propenso a overfitting
- ✅ No requiere normalización de datos

### Hiperparámetros importantes:
- **n_estimators**: Número de árboles (más árboles = más preciso pero más lento)
- **max_depth**: Profundidad máxima de cada árbol (controla overfitting)
- **random_state**: Semilla para reproducibilidad

### Train/Test Split:
Dividimos los datos en:
- **75% Training**: Para entrenar el modelo
- **25% Testing**: Para evaluar el modelo (datos que nunca ha visto)

### ¿Qué es Accuracy?
**Accuracy** (precisión) = % de predicciones correctas
- Fórmula: (Aciertos / Total) × 100
- Ejemplo: 90/100 = 90% accuracy

In [ ]:
# CELDA 8: ENTRENAMIENTO DEL MODELO RANDOM FOREST
# Aquí entrenamos nuestro modelo de Machine Learning

print("🧠 ENTRENANDO MODELO DE MACHINE LEARNING...")
print("="*80)

# ==========================================
# 1. PREPARAR FEATURES (X) Y LABELS (y)
# ==========================================
print("\n📊 Preparando datos...")

# Definir qué columnas usar como features (características)
# Estas son las variables que el modelo usará para predecir
feature_columns = [
    'attack_diff', 'defense_diff', 'form_diff', 'points_diff', 'goal_diff_diff',
    'teamA_attack', 'teamA_defense', 'teamA_form',
    'teamB_attack', 'teamB_defense', 'teamB_form',
    'score_diff'
]

# X: matriz de features (lo que el modelo usa para predecir)
# y: vector de labels (lo que queremos predecir: winner)
X = df_matches[feature_columns]
y = df_matches['winner']

print(f"   - Features (X): {X.shape[0]} filas × {X.shape[1]} columnas")
print(f"   - Labels (y): {len(y)} valores")

# ==========================================
# 2. DIVIDIR EN TRAIN Y TEST
# ==========================================
print("\n✂️ Dividiendo datos en entrenamiento y prueba...")

# train_test_split: función que divide aleatoriamente los datos
# test_size=0.25: 25% para testing, 75% para training
# random_state=42: semilla aleatoria (para reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25, 
    random_state=42
)

print(f"   - Training set: {len(X_train)} muestras ({len(X_train)/len(X)*100:.1f}%)")
print(f"   - Testing set: {len(X_test)} muestras ({len(X_test)/len(X)*100:.1f}%)")
print(f"   - Total features: {len(feature_columns)}")

# ==========================================
# 3. CREAR Y ENTRENAR EL MODELO
# ==========================================
print("\n🌲 Creando Random Forest Classifier...")

# Crear instancia del modelo
# n_estimators=100: usar 100 árboles de decisión
# random_state=42: reproducibilidad
# max_depth=10: profundidad máxima de árboles (evita overfitting)
rf_model = RandomForestClassifier(
    n_estimators=100,  # Número de árboles en el bosque
    random_state=42,   # Semilla para reproducibilidad
    max_depth=10       # Profundidad máxima de cada árbol
)

print("   - Número de árboles: 100")
print("   - Profundidad máxima: 10")
print("\n🔄 Entrenando modelo (esto puede tomar 10-20 segundos)...")

# .fit(): entrenar el modelo con datos de training
# El modelo "aprende" los patrones en los datos
rf_model.fit(X_train, y_train)

print("✅ Modelo entrenado exitosamente")

# ==========================================
# 4. HACER PREDICCIONES Y EVALUAR
# ==========================================
print("\n🎯 Evaluando modelo en test set...")

# .predict(): hacer predicciones en datos de test
y_pred = rf_model.predict(X_test)

# accuracy_score: calcular precisión (% de aciertos)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n📊 RESULTADO:")
print(f"   🎯 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   ✅ Predicciones correctas: {(y_pred == y_test).sum()} / {len(y_test)}")
print(f"   ❌ Predicciones incorrectas: {(y_pred != y_test).sum()} / {len(y_test)}")

# Interpretación del accuracy
if accuracy >= 0.90:
    print("\n🏆 ¡Excelente! El modelo es muy preciso")
elif accuracy >= 0.80:
    print("\n👍 Muy bueno! El modelo es bastante confiable")
elif accuracy >= 0.70:
    print("\n👌 Aceptable. El modelo funciona razonablemente bien")
else:
    print("\n⚠️ El modelo podría mejorar")

# ==========================================
# 5. GUARDAR MODELO ENTRENADO
# ==========================================
print("\n💾 Guardando modelo...")

# pickle: librería para serializar objetos de Python
# 'wb': write binary (escribir en modo binario)
with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("✅ Modelo guardado en: models/best_model.pkl")
print("\n📌 NOTA: Ahora puedes cargar este modelo sin necesidad de reentrenarlo")

## 📈 SECCIÓN 9: EVALUACIÓN AVANZADA DEL MODELO

### Métricas de evaluación:

#### 1. **Confusion Matrix (Matriz de Confusión)**:
Tabla 2×2 que muestra:
- **True Positives (TP)**: Predijo A gana y acertó
- **True Negatives (TN)**: Predijo B gana y acertó
- **False Positives (FP)**: Predijo A gana pero perdió
- **False Negatives (FN)**: Predijo B gana pero perdió

#### 2. **ROC Curve (Receiver Operating Characteristic)**:
Gráfico que muestra el rendimiento del clasificador:
- **Eje X**: False Positive Rate (tasa de falsos positivos)
- **Eje Y**: True Positive Rate (tasa de verdaderos positivos)
- Cuanto más cerca de la esquina superior izquierda, mejor

#### 3. **AUC Score (Area Under Curve)**:
Área bajo la curva ROC:
- **1.0**: Clasificador perfecto
- **0.9-1.0**: Excelente
- **0.8-0.9**: Muy bueno
- **0.7-0.8**: Bueno
- **0.5-0.7**: Pobre
- **0.5**: Random (como lanzar una moneda)

In [ ]:
# CELDA 9: EVALUACIÓN AVANZADA DEL MODELO
# Calculamos métricas adicionales y creamos visualizaciones

print("📈 EVALUACIÓN AVANZADA DEL MODELO...")
print("="*80)

# ==========================================
# 1. CALCULAR MÉTRICAS
# ==========================================
print("\n📊 Calculando métricas...")

# Confusion Matrix
# Matriz 2x2 con TP, TN, FP, FN
cm = confusion_matrix(y_test, y_pred)
print(f"   ✅ Confusion Matrix calculada")

# Probabilidades predichas (para curva ROC)
# predict_proba: devuelve probabilidades para cada clase
# [:, 1]: seleccionar probabilidades de la clase 1 (Team A gana)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# ROC Curve
# fpr: False Positive Rate
# tpr: True Positive Rate
# thresholds: umbrales de decisión
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# AUC: Area Under Curve
roc_auc = auc(fpr, tpr)
print(f"   ✅ ROC Curve calculada (AUC: {roc_auc:.4f})")

# ==========================================
# 2. CREAR VISUALIZACIONES
# ==========================================
print("\n📊 Generando gráficos de evaluación...")

# Crear figura con 2 subgráficos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- GRÁFICO 1: CONFUSION MATRIX ---
# Usar seaborn para crear un heatmap bonito
sns.heatmap(
    cm,                                          # Datos de la matriz
    annot=True,                                  # Mostrar números
    fmt='d',                                     # Formato entero
    cmap='Blues',                                # Paleta azul
    ax=axes[0],                                  # Primer subplot
    xticklabels=['Team B Gana', 'Team A Gana'],  # Etiquetas X
    yticklabels=['Team B Gana', 'Team A Gana'],  # Etiquetas Y
    cbar_kws={'label': 'Cantidad'}               # Etiqueta colorbar
)
axes[0].set_xlabel('Predicción', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Valor Real', fontsize=11, fontweight='bold')
axes[0].set_title('Matriz de Confusión', fontsize=13, fontweight='bold')

# Agregar texto explicativo
axes[0].text(
    0.5, -0.15,
    'Diagonal principal = Predicciones correctas',
    ha='center',
    transform=axes[0].transAxes,
    fontsize=9,
    style='italic'
)

# --- GRÁFICO 2: ROC CURVE ---
# Graficar curva ROC
axes[1].plot(
    fpr, tpr,
    color='darkorange',
    lw=2,
    label=f'ROC Curve (AUC = {roc_auc:.2f})'  # Leyenda con AUC
)

# Línea diagonal (clasificador aleatorio)
axes[1].plot(
    [0, 1], [0, 1],
    color='navy',
    lw=2,
    linestyle='--',
    label='Random Classifier (AUC = 0.50)'
)

# Configurar ejes y etiquetas
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
axes[1].set_title('Curva ROC', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

# Guardar y mostrar
plt.tight_layout()
plt.savefig('outputs/evaluacion_modelo.png', dpi=150, bbox_inches='tight')
plt.show()

# ==========================================
# 3. RESUMEN DE MÉTRICAS
# ==========================================
print("\n📊 RESUMEN DE MÉTRICAS:")
print("="*80)
print(f"   🎯 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   📈 AUC Score: {roc_auc:.4f}")
print(f"\n   Confusion Matrix:")
print(f"      True Negatives:  {cm[0,0]} (predijo B gana, acertó)")
print(f"      False Positives: {cm[0,1]} (predijo A gana, falló)")
print(f"      False Negatives: {cm[1,0]} (predijo B gana, falló)")
print(f"      True Positives:  {cm[1,1]} (predijo A gana, acertó)")

print(f"\n💾 Gráficos guardados en: outputs/evaluacion_modelo.png")

# Interpretación del AUC
print("\n💡 INTERPRETACIÓN:")
if roc_auc >= 0.9:
    print("   🏆 AUC excelente! El modelo discrimina muy bien entre clases")
elif roc_auc >= 0.8:
    print("   👍 AUC muy bueno! El modelo es confiable")
elif roc_auc >= 0.7:
    print("   👌 AUC aceptable. El modelo funciona bien")
else:
    print("   ⚠️ AUC podría mejorar")

## 🎯 SECCIÓN 10: SISTEMA DE PREDICCIÓN

### ¿Cómo funciona la función predict_match()?

1. **Recibe**: IDs de dos equipos (team_a_id, team_b_id)
2. **Busca**: Estadísticas de ambos equipos en df_features
3. **Calcula**: Las 12 características diferenciales
4. **Predice**: Usa el modelo entrenado para calcular probabilidades
5. **Retorna**: Diccionario con resultados completos

### ¿Qué son las probabilidades?
- **predict_proba()**: devuelve probabilidades para cada clase
- proba[0]: probabilidad de que gane Team B (clase 0)
- proba[1]: probabilidad de que gane Team A (clase 1)
- Las probabilidades suman 100%

### Ejemplo:
Si predict_proba devuelve [0.30, 0.70]:
- Team B: 30% probabilidad
- Team A: 70% probabilidad
- Ganador: Team A (mayor probabilidad)
- Confianza: 70%

In [ ]:
# CELDA 10: SISTEMA DE PREDICCIÓN
# Función para predecir resultado de cualquier partido

print("🎯 CREANDO SISTEMA DE PREDICCIÓN...")
print("="*80)

def predict_match(team_a_id, team_b_id):
    """
    Predice el resultado de un partido entre dos equipos.
    
    Parámetros:
    -----------
    team_a_id : int
        ID del primer equipo (equipo local)
    team_b_id : int
        ID del segundo equipo (equipo visitante)
    
    Retorna:
    --------
    dict
        Diccionario con:
        - teamA: nombre del equipo A
        - teamB: nombre del equipo B
        - probA: probabilidad de victoria de A (%)
        - probB: probabilidad de victoria de B (%)
        - winner: nombre del ganador predicho
        - confidence: confianza de la predicción (%)
    """
    # 1. Buscar información de ambos equipos
    # .iloc[0]: tomar primera fila (debería haber solo una)
    team_a = df_features[df_features['id'] == team_a_id].iloc[0]
    team_b = df_features[df_features['id'] == team_b_id].iloc[0]
    
    # 2. Crear DataFrame con las características del partido
    # IMPORTANTE: Usar las MISMAS 12 características que en el entrenamiento
    match = pd.DataFrame([{
        'attack_diff': team_a['attack_strength'] - team_b['attack_strength'],
        'defense_diff': team_a['defense_strength'] - team_b['defense_strength'],
        'form_diff': team_a['form_score'] - team_b['form_score'],
        'points_diff': team_a['points'] - team_b['points'],
        'goal_diff_diff': team_a['goalDifference'] - team_b['goalDifference'],
        'teamA_attack': team_a['attack_strength'],
        'teamA_defense': team_a['defense_strength'],
        'teamA_form': team_a['form_score'],
        'teamB_attack': team_b['attack_strength'],
        'teamB_defense': team_b['defense_strength'],
        'teamB_form': team_b['form_score'],
        'score_diff': team_a['team_score'] - team_b['team_score'],
    }])
    
    # 3. Hacer predicción con el modelo
    # predict_proba: devuelve probabilidades [prob_B, prob_A]
    proba = rf_model.predict_proba(match)[0]
    prob_b = proba[0] * 100  # Convertir a porcentaje
    prob_a = proba[1] * 100  # Convertir a porcentaje
    
    # 4. Determinar ganador (el que tenga mayor probabilidad)
    winner_name = team_a['name'] if prob_a > prob_b else team_b['name']
    
    # 5. Confianza = probabilidad del ganador
    confidence = max(prob_a, prob_b)
    
    # 6. Retornar resultados
    return {
        'teamA': team_a['name'],
        'teamB': team_b['name'],
        'probA': round(prob_a, 2),
        'probB': round(prob_b, 2),
        'winner': winner_name,
        'confidence': round(confidence, 2)
    }

# ==========================================
# PRUEBA DE LA FUNCIÓN
# ==========================================
print("\n✅ Función predict_match() creada")
print("\n🧪 PRUEBA DEL SISTEMA:")
print("="*80)

# Hacer predicción de ejemplo con primeros 2 equipos
example = predict_match(
    df_features.iloc[0]['id'],  # Primer equipo (mejor clasificado)
    df_features.iloc[1]['id']   # Segundo equipo
)

# Mostrar resultado
print(f"\n⚽ PARTIDO DE EJEMPLO:")
print(f"   {example['teamA']} vs {example['teamB']}")
print(f"\n📊 PREDICCIÓN:")
print(f"   🏆 Ganador predicho: {example['winner']}")
print(f"   💪 Confianza: {example['confidence']}%")
print(f"\n📈 PROBABILIDADES:")
print(f"   {example['teamA']}: {example['probA']}%")
print(f"   {example['teamB']}: {example['probB']}%")

# Validación
total_prob = example['probA'] + example['probB']
print(f"\n✅ Suma de probabilidades: {total_prob:.2f}% (debe ser 100%)")

print("\n🎉 Sistema de predicción listo para usar!")

## 🌐 SECCIÓN 11: INTERFAZ INTERACTIVA CON GRADIO

### ¿Qué es Gradio?
**Gradio** es una librería de Python que permite crear **interfaces web interactivas** para modelos de ML en minutos.

### Ventajas de Gradio:
- ✅ No requiere conocimientos de HTML/CSS/JavaScript
- ✅ Crea UI automáticamente
- ✅ Genera URL pública para compartir
- ✅ Funciona en Colab/Jupyter

### Componentes que usaremos:
- **gr.Dropdown**: Menú desplegable para seleccionar equipos
- **gr.Markdown**: Para mostrar resultados formateados
- **gr.Interface**: Contenedor principal de la aplicación

### ¿Cómo funciona?
1. Usuario selecciona 2 equipos de los dropdowns
2. Hace clic en "Submit"
3. Se llama a gradio_predict()
4. Se muestra el resultado en Markdown

### share=True:
Cuando usamos `share=True`, Gradio crea una **URL pública temporal** que cualquiera puede abrir (dura 72 horas).

In [ ]:
# CELDA 11: INTERFAZ INTERACTIVA CON GRADIO
# Creamos una interfaz web para que usuarios puedan hacer predicciones fácilmente

print("🌐 CREANDO INTERFAZ WEB INTERACTIVA...")
print("="*80)

# Importar Gradio
import gradio as gr

def gradio_predict(team_a_name, team_b_name):
    """
    Función adaptadora para Gradio.
    Recibe NOMBRES de equipos (strings) en lugar de IDs.
    
    Parámetros:
    -----------
    team_a_name : str
        Nombre del equipo A
    team_b_name : str
        Nombre del equipo B
    
    Retorna:
    --------
    str
        Texto en formato Markdown con el resultado
    """
    try:
        # 1. Validar que ambos equipos fueron seleccionados
        if not team_a_name or not team_b_name:
            return "⚠️ Por favor selecciona ambos equipos"
        
        # 2. Validar que sean equipos diferentes
        if team_a_name == team_b_name:
            return "❌ Debes seleccionar dos equipos diferentes"
        
        # 3. Buscar equipos en el DataFrame por nombre
        team_a = df_features[df_features['name'] == team_a_name]
        team_b = df_features[df_features['name'] == team_b_name]
        
        # 4. Verificar que los equipos existan
        if team_a.empty or team_b.empty:
            return "❌ Equipo no encontrado en la base de datos"
        
        # 5. Obtener IDs y hacer predicción
        team_a_id = team_a.iloc[0]['id']
        team_b_id = team_b.iloc[0]['id']
        
        # Llamar a nuestra función de predicción
        result = predict_match(team_a_id, team_b_id)
        
        # 6. Formatear resultado en Markdown
        # Markdown permite usar formato rico (negritas, listas, etc)
        output = f"""
## 🏆 PREDICCIÓN DE PARTIDO

### {result['teamA']} 🆚 {result['teamB']}

---

**🎯 Ganador Predicho:** {result['winner']}

**📊 Confianza:** {result['confidence']}%

### Probabilidades de Victoria:
- **{result['teamA']}:** {result['probA']}%
- **{result['teamB']}:** {result['probB']}%

---

### 📈 Estadísticas Comparativas:

**{result['teamA']}:**
- ⚔️ Ataque: {team_a.iloc[0]['attack_strength']:.2f}
- 🛡️ Defensa: {team_a.iloc[0]['defense_strength']:.2f}
- 📊 Forma: {team_a.iloc[0]['form_score']:.2f}
- ⭐ Score Total: {team_a.iloc[0]['team_score']:.2f}

**{result['teamB']}:**
- ⚔️ Ataque: {team_b.iloc[0]['attack_strength']:.2f}
- 🛡️ Defensa: {team_b.iloc[0]['defense_strength']:.2f}
- 📊 Forma: {team_b.iloc[0]['form_score']:.2f}
- ⭐ Score Total: {team_b.iloc[0]['team_score']:.2f}

---

🤖 **Modelo:** Random Forest (Accuracy: {accuracy:.2%})

⚠️ *Esta es una predicción educativa basada en estadísticas actuales de la temporada. No debe usarse para apuestas reales.*
        """
        
        return output
    
    except Exception as e:
        # Si hay cualquier error, mostrar mensaje amigable
        return f"❌ Error inesperado: {str(e)}"

# ==========================================
# CREAR INTERFAZ GRADIO
# ==========================================
print("\n🎨 Configurando interfaz...")

# Obtener lista de nombres de equipos ordenada alfabéticamente
team_names = sorted(df_features['name'].tolist())

# Crear interfaz con gr.Interface
iface = gr.Interface(
    # Función que se ejecuta cuando el usuario hace clic en Submit
    fn=gradio_predict,
    
    # Inputs: dos dropdowns
    inputs=[
        gr.Dropdown(
            choices=team_names,              # Lista de equipos
            label="🔵 Selecciona Equipo A",  # Etiqueta
            value=team_names[0] if team_names else None  # Valor por defecto
        ),
        gr.Dropdown(
            choices=team_names,
            label="🔴 Selecciona Equipo B",
            value=team_names[1] if len(team_names) > 1 else None
        )
    ],
    
    # Output: área de texto con formato Markdown
    outputs=gr.Markdown(label="📊 Resultado de la Predicción"),
    
    # Configuración de la interfaz
    title="⚽ Premier League Match Predictor",
    description="""Selecciona dos equipos para predecir el resultado del partido usando Machine Learning (Random Forest).
    
El modelo analiza estadísticas actuales de la temporada para calcular probabilidades de victoria.""",
    
    theme="soft",  # Tema visual
    
    # Ejemplos predefinidos (clic rápido)
    examples=[
        [team_names[0], team_names[1]],
        [team_names[2], team_names[3]],
    ] if len(team_names) >= 4 else None,
    
    allow_flagging="never"  # Desactivar botón de flag
)

print("\n✅ Interfaz creada exitosamente")
print("\n🚀 LANZANDO INTERFAZ GRADIO...")
print("="*80)
print("\n⏳ Generando URL pública (puede tomar 30-60 segundos)...")
print("\n📌 INSTRUCCIONES:")
print("   1. Espera a que aparezca la URL pública")
print("   2. Haz clic en el link para abrir la interfaz")
print("   3. Selecciona dos equipos")
print("   4. Haz clic en Submit")
print("   5. ¡Observa la predicción!")
print("\n⚠️ NOTA: La URL pública es temporal (dura 72 horas)")
print("="*80)

# Lanzar interfaz
# share=True: crear URL pública
# debug=False: no mostrar mensajes de debug
iface.launch(share=True, debug=False)

## 💾 SECCIÓN 12: EXPORTACIÓN DE ARCHIVOS CSV

### ¿Por qué exportar a CSV?
CSV (Comma-Separated Values) es un formato universal para datos tabulares:
- ✅ Se abre en Excel, Google Sheets, etc.
- ✅ Fácil de compartir
- ✅ Ligero y portable
- ✅ Compatible con cualquier lenguaje de programación

### Archivos que exportaremos:
1. **teams_analysis.csv**: Análisis completo de equipos con features
2. **match_predictions.csv**: Predicciones de partidos entre top equipos
3. **project_summary.csv**: Resumen de métricas del proyecto

### Estructura típica de CSV:
```
columna1,columna2,columna3
valor1,valor2,valor3
valor4,valor5,valor6
```

In [ ]:
# CELDA 12: EXPORTACIÓN DE CSVs
# Generamos archivos CSV para compartir/entregar resultados

print("💾 EXPORTANDO ARCHIVOS CSV...")
print("="*80)

# ==========================================
# CSV 1: ANÁLISIS DE EQUIPOS
# ==========================================
print("\n📊 Generando teams_analysis.csv...")

# Seleccionar columnas relevantes para el análisis
output_teams = df_features[[
    'id', 'name', 'position', 'points', 'played',
    'won', 'draw', 'lost',
    'goalsFor', 'goalsAgainst', 'goalDifference',
    'attack_strength', 'defense_strength', 'form_score', 'team_score'
]]

# Ordenar por team_score (mejores primero)
output_teams = output_teams.sort_values('team_score', ascending=False)

# Guardar CSV
output_teams.to_csv('outputs/teams_analysis.csv', index=False)
print(f"   ✅ Guardado: {len(output_teams)} equipos")

# ==========================================
# CSV 2: PREDICCIONES DE PARTIDOS
# ==========================================
print("\n🎯 Generando match_predictions.csv...")

predictions = []  # Lista para almacenar predicciones

# Generar predicciones para top 6 equipos entre sí
# (6 equipos = 30 partidos posibles)
n_teams = min(6, len(df_features))

for i in range(n_teams):
    for j in range(i+1, n_teams):  # i+1 para evitar duplicados
        # Hacer predicción
        pred = predict_match(
            df_features.iloc[i]['id'],
            df_features.iloc[j]['id']
        )
        
        # Agregar a lista
        predictions.append(pred)

# Convertir a DataFrame y guardar
df_predictions = pd.DataFrame(predictions)
df_predictions.to_csv('outputs/match_predictions.csv', index=False)
print(f"   ✅ Guardado: {len(predictions)} predicciones")

# ==========================================
# CSV 3: RESUMEN DEL PROYECTO
# ==========================================
print("\n📋 Generando project_summary.csv...")

# Crear DataFrame con métricas clave
summary = pd.DataFrame({
    'metric': [
        'Total Teams',
        'Total Matches in Training',
        'Best Model',
        'Best Model Accuracy',
        'Total Features',
        'Training Samples',
        'Test Samples',
        'AUC Score'
    ],
    'value': [
        len(df_features),
        len(df_matches),
        'Random Forest',
        f"{accuracy:.4f}",
        len(feature_columns),
        len(X_train),
        len(X_test),
        f"{roc_auc:.4f}"
    ]
})

summary.to_csv('outputs/project_summary.csv', index=False)
print(f"   ✅ Guardado: {len(summary)} métricas")

# ==========================================
# CONFIRMACIÓN
# ==========================================
print("\n🎉 TODOS LOS ARCHIVOS CSV EXPORTADOS EXITOSAMENTE")
print("\n📁 Archivos generados en outputs/:")
print("   - teams_analysis.csv")
print("   - match_predictions.csv")
print("   - project_summary.csv")

print("\n💡 Para descargar en Colab:")
print("   1. Haz clic en la carpeta 📁 en el panel izquierdo")
print("   2. Navega a 'outputs/'")
print("   3. Click derecho en el archivo → Download")

## 🎉 SECCIÓN 13: RESUMEN FINAL DEL PROYECTO

### ¿Qué hemos logrado?

En este notebook hemos construido un sistema completo de predicción de partidos de fútbol:

#### 1. **Obtención de Datos**:
- Conectamos a una API real (football-data.org)
- Obtuvimos datos actualizados de la Premier League

#### 2. **Análisis Exploratorio**:
- Calculamos estadísticas descriptivas
- Identificamos patrones y tendencias
- Creamos visualizaciones informativas

#### 3. **Feature Engineering**:
- Creamos 12 características derivadas
- Normalizamos métricas por partido
- Construimos scores compuestos

#### 4. **Machine Learning**:
- Entrenamos un modelo Random Forest
- Evaluamos con métricas robustas (Accuracy, AUC)
- Optimizamos hiperparámetros

#### 5. **Interfaz Interactiva**:
- Creamos una UI web con Gradio
- Generamos URL pública para compartir
- Facilitamos el uso para no-programadores

#### 6. **Exportación de Resultados**:
- Generamos CSVs profesionales
- Guardamos visualizaciones
- Documentamos todo el proceso

### Próximos pasos sugeridos:
- Probar con más modelos (XGBoost, Neural Networks)
- Agregar más features (histórico de enfrentamientos)
- Integrar datos de lesiones/suspensiones
- Considerar factor de localía (local/visitante)
- Hacer seguimiento de predicciones vs resultados reales

In [ ]:
# CELDA 13: RESUMEN FINAL
# Mostramos un resumen completo de todo lo realizado

print("\n" + "="*80)
print(" " * 28 + "RESUMEN FINAL DEL PROYECTO")
print("="*80)

# ==========================================
# ESTADÍSTICAS DEL PROYECTO
# ==========================================
print("\n📊 ESTADÍSTICAS DEL PROYECTO:")
print("-" * 80)
print(f"\n🏆 Datos Procesados:")
print(f"   - Equipos analizados: {len(df_features)}")
print(f"   - Partidos en training: {len(df_matches)}")
print(f"   - Features creados: {len(df_features.columns) - len(df_teams.columns)}")
print(f"   - Predicciones generadas: {len(predictions)}")

print(f"\n🤖 Modelo de Machine Learning:")
print(f"   - Algoritmo: Random Forest")
print(f"   - Número de árboles: 100")
print(f"   - Features utilizados: {len(feature_columns)}")
print(f"   - Training samples: {len(X_train)}")
print(f"   - Test samples: {len(X_test)}")

print(f"\n🎯 Métricas de Rendimiento:")
print(f"   - Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   - AUC Score: {roc_auc:.4f}")
print(f"   - True Positives: {cm[1,1]}")
print(f"   - True Negatives: {cm[0,0]}")
print(f"   - False Positives: {cm[0,1]}")
print(f"   - False Negatives: {cm[1,0]}")

# ==========================================
# ARCHIVOS GENERADOS
# ==========================================
print(f"\n📁 ARCHIVOS GENERADOS:")
print("-" * 80)

print("\n   📂 data/ (Datos procesados):")
print("      ├── teams_raw.csv")
print("      ├── teams_with_features.csv")
print("      └── matches_dataset.csv")

print("\n   📂 outputs/ (Resultados):")
print("      ├── teams_analysis.csv")
print("      ├── match_predictions.csv")
print("      ├── project_summary.csv")
print("      ├── visualizaciones_principales.png")
print("      └── evaluacion_modelo.png")

print("\n   📂 models/ (Modelos ML):")
print("      └── best_model.pkl")

# ==========================================
# INSIGHTS CLAVE
# ==========================================
print("\n💡 INSIGHTS CLAVE DEL ANÁLISIS:")
print("-" * 80)

# Mejor equipo por score
best_team = df_features.loc[df_features['team_score'].idxmax()]
print(f"\n🏆 Mejor equipo (por Team Score):")
print(f"   {best_team['name']} - Score: {best_team['team_score']:.2f}")

# Mejor ataque
best_attack = df_features.loc[df_features['goalsFor'].idxmax()]
print(f"\n⚔️ Mejor ataque:")
print(f"   {best_attack['name']} - {best_attack['goalsFor']} goles")

# Mejor defensa
best_defense = df_features.loc[df_features['goalsAgainst'].idxmin()]
print(f"\n🛡️ Mejor defensa:")
print(f"   {best_defense['name']} - {best_defense['goalsAgainst']} goles en contra")

# Predicción más confiada
most_confident = df_predictions.loc[df_predictions['confidence'].idxmax()]
print(f"\n🎯 Predicción más confiada:")
print(f"   {most_confident['teamA']} vs {most_confident['teamB']}")
print(f"   Ganador: {most_confident['winner']} ({most_confident['confidence']}%)")

# ==========================================
# MENSAJE FINAL
# ==========================================
print("\n" + "="*80)
print("\n✅ PROYECTO COMPLETADO EXITOSAMENTE")
print("\n🌐 Interfaz Gradio disponible arriba ⬆️")
print("💾 Para descargar archivos: Panel izquierdo → Carpeta 📁 → outputs/")
print("\n📚 Tecnologías utilizadas:")
print("   - Python 3.x")
print("   - pandas, numpy, matplotlib, seaborn")
print("   - scikit-learn (Random Forest)")
print("   - Gradio (interfaz web)")
print("   - football-data.org API")

print("\n🎓 Conceptos aprendidos:")
print("   ✓ Obtención de datos desde APIs")
print("   ✓ Análisis exploratorio de datos (EDA)")
print("   ✓ Feature Engineering")
print("   ✓ Machine Learning (clasificación)")
print("   ✓ Evaluación de modelos")
print("   ✓ Creación de interfaces web")
print("   ✓ Exportación y visualización de resultados")

print("\n" + "="*80)
print("\n🙏 ¡Gracias por usar este notebook!")
print("\n⚽ ¡Disfruta prediciendo partidos de la Premier League!")
print("\n" + "="*80)